In [ ]:
#@title 1. Options
SOURCE_ARCHIVE = "/content/drive/MyDrive/humanoid/humanoid-source.tar.gz" #@param {type:"string"}
TASK = "G1-Open-HeightMap" #@param ["G1-Flat-HeightMap", "G1-Rough-HeightMap", "G1-Open-HeightMap", "G1-Open-Maze-HeightMap", "Unitree-G1-Flat", "Unitree-G1-Rough", "G1-Comp-Rough", "G1-Comp-Course", "G1-Comp-Crouch-Test"]
START_TERRAIN_LEVEL = -1 #@param {type:"integer"}  # -1 keeps the task default; 0–9 starts every robot on that row
NUM_ENVS = 1024 #@param {type:"integer"}
ITERATIONS = 2000 #@param {type:"integer"}
ROLLOUT_STEPS = 24 #@param {type:"integer"}
NUM_MINIBATCHES = 4 #@param {type:"integer"}
SAVE_INTERVAL = 200 #@param {type:"integer"}
RUN_NAME = "open_heightmap_from_5996" #@param {type:"string"}
WANDB_PROJECT = "humanoid-locomotion" #@param {type:"string"}
WANDB_ENTITY = "" #@param {type:"string"}

from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path("/content/humanoid")
PYTHON = "/content/mjlab-venv/bin/python"
assert all(n > 0 for n in (NUM_ENVS, ITERATIONS, ROLLOUT_STEPS, NUM_MINIBATCHES, SAVE_INTERVAL))
assert (NUM_ENVS * ROLLOUT_STEPS) % NUM_MINIBATCHES == 0
assert START_TERRAIN_LEVEL == -1 or (0 <= START_TERRAIN_LEVEL <= 9 and TASK in {"G1-Open-HeightMap", "G1-Open-Maze-HeightMap"}), "Start level is supported for the Open height-map tasks."
print(f"{NUM_ENVS:,} robots; {ITERATIONS:,} updates; {NUM_ENVS * ROLLOUT_STEPS:,} transitions/update")


In [ ]:
# 2. Mount Drive and load the repository snapshot made on your Mac.
from google.colab import drive
import shutil

drive.mount("/content/drive")
archive = Path(SOURCE_ARCHIVE)
assert archive.is_file(), f"Upload your source archive first: {archive}"
entries = subprocess.check_output(["tar", "-tzf", str(archive)], text=True).splitlines()
assert entries and all(
    entry == "humanoid" or (
        entry.startswith("humanoid/")
        and ".." not in Path(entry).parts
    )
    for entry in entries
), "Archive must contain a single top-level humanoid/ folder."
assert any(entry.startswith("humanoid/training/unitree_rl_mjlab/src/") for entry in entries), (
    "Archive is missing the populated Unitree submodule."
)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["tar", "-xzf", str(archive), "-C", str(REPO_DIR.parent)], check=True)
print("Loaded source archive:", archive)


In [ ]:
# 3. Install in an isolated venv using the same pins as the team's Docker setup.
# The requirements filename says native, but pip selects Linux/CUDA wheels here.
import time
assert (3, 10) <= sys.version_info[:2] < (3, 14), "Use Python 3.10–3.13 with these pins."
# Colab's Python version may not have a matching apt venv package.
venv_dir = Path(PYTHON).parent.parent
venv_ready = Path(PYTHON).is_file() and subprocess.run([PYTHON, "-m", "pip", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not venv_ready:
    print("Creating the Python environment...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", sys.executable, "--seed", "--clear", str(venv_dir)], check=True)
# RSL-RL passes start_method to W&B; W&B 0.29 removed that setting.
print("Installing training dependencies; pip output appears below.", flush=True)
install = subprocess.Popen(
    [PYTHON, "-u", "-m", "pip", "install", "-v", "--progress-bar", "raw",
     "-r", str(REPO_DIR / "docs/requirements-native-mjlab.txt"), "wandb<0.29"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
started = time.monotonic()
try:
    for line in install.stdout:
        print(line, end="", flush=True)
    return_code = install.wait()
except KeyboardInterrupt:
    install.terminate()
    install.wait()
    raise
if return_code:
    raise subprocess.CalledProcessError(return_code, install.args)
print(f"Dependencies installed in {time.monotonic() - started:.0f}s.", flush=True)


In [ ]:
# 4. Mount Drive and verify CUDA for both PyTorch and Warp.
from google.colab import drive
drive.mount("/content/drive")
WORK_DIR = Path("/content/drive/MyDrive/humanoid")
WORK_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([PYTHON, "-c", "import torch, warp as wp; wp.init(); assert torch.cuda.is_available() and wp.is_cuda_available(), 'CUDA unavailable'; print(torch.cuda.get_device_name(0)); wp.zeros(1, device='cuda:0'); wp.synchronize_device('cuda:0')"], check=True)
print("Outputs will be saved under:", WORK_DIR / "logs/rsl_rl")


In [ ]:
# 5. Pick a bundled checkpoint or browse a previous run in Drive.
import ipywidgets as widgets
from IPython.display import display

LOG_ROOT = WORK_DIR / "logs/rsl_rl"
HEIGHTMAP_TASKS = {"G1-Flat-HeightMap", "G1-Rough-HeightMap", "G1-Open-HeightMap", "G1-Open-Maze-HeightMap"}
HEIGHTMAP_EXPERIMENTS = {"g1_flat_heightmap", "g1_rough_heightmap", "g1_open_heightmap", "g1_open_maze_heightmap"}
bundled_models = sorted([
    *(REPO_DIR / "training/models").glob("best_g1_*_heightmap/model_*.pt"),
    *(REPO_DIR / "training/models/g1_open_heightmap").glob("model_*.pt"),
])
rough_model = REPO_DIR / "training/models/best_g1_rough_heightmap/model_5996.pt"
open_model = REPO_DIR / "training/models/g1_open_heightmap/model_6995.pt"
preferred_model = open_model if TASK in {"G1-Open-HeightMap", "G1-Open-Maze-HeightMap"} and open_model in bundled_models else rough_model

resume_mode = widgets.Checkbox(value=TASK in {"G1-Open-HeightMap", "G1-Open-Maze-HeightMap"}, description="Start from checkpoint")
source_location = widgets.Dropdown(
    options=[("Repository model", "repo"), ("Drive run", "drive")],
    value="repo" if TASK in {"G1-Open-HeightMap", "G1-Open-Maze-HeightMap"} else "drive",
    description="Checkpoint source:", style={"description_width": "initial"},
)
repo_model = widgets.Dropdown(
    options=[(str(p.relative_to(REPO_DIR)), str(p)) for p in bundled_models],
    value=str(preferred_model) if preferred_model in bundled_models else None,
    description="Repository model:", style={"description_width": "initial"},
    layout=widgets.Layout(width="95%"),
)
source_environment = widgets.Dropdown(description="Source env:", style={"description_width": "initial"})
source_run = widgets.Dropdown(description="Source run:", layout=widgets.Layout(width="95%"))
source_model = widgets.Dropdown(description="Model:")
repo_browser = widgets.VBox([repo_model])
drive_browser = widgets.VBox([source_environment, source_run, source_model])
checkpoint_browser = widgets.VBox([source_location, repo_browser, drive_browser])

def update_models(change=None):
    folder = LOG_ROOT / source_environment.value / source_run.value if source_environment.value and source_run.value else None
    models = list(folder.glob("model_*.pt")) if folder else []
    models.sort(key=lambda p: (int(p.stem[6:]) if p.stem[6:].isdigit() else -1, p.name), reverse=True)
    source_model.options = [(p.name, str(p)) for p in models]
    source_model.value = str(models[0]) if models else None

def update_runs(change=None):
    folder = LOG_ROOT / source_environment.value if source_environment.value else None
    source_run.options = sorted((p.name for p in folder.iterdir() if p.is_dir() and any(p.glob("model_*.pt"))), reverse=True) if folder else []
    source_run.value = source_run.options[0] if source_run.options else None
    update_models()

def update_mode(change=None):
    checkpoint_browser.layout.display = "" if resume_mode.value else "none"
    repo_browser.layout.display = "" if source_location.value == "repo" else "none"
    drive_browser.layout.display = "" if source_location.value == "drive" else "none"

source_environment.observe(update_runs, names="value")
source_run.observe(update_models, names="value")
source_location.observe(update_mode, names="value")
resume_mode.observe(update_mode, names="value")
folders = (p for p in LOG_ROOT.iterdir() if p.is_dir() and any(p.glob("*/model_*.pt"))) if LOG_ROOT.exists() else ()
source_environment.options = sorted(
    p.name for p in folders if TASK not in HEIGHTMAP_TASKS or p.name in HEIGHTMAP_EXPERIMENTS
)
source_environment.value = source_environment.options[0] if source_environment.options else None
update_runs()
update_mode()
display(resume_mode, checkpoint_browser)
print(f"Training environment: {TASK}. Iterations are additional updates when resuming.")
print("The repository models are compatible with the G1-*-HeightMap tasks only.")
if not source_environment.options:
    print("No compatible Drive checkpoints found under:", LOG_ROOT)


In [ ]:
# 6. Train with W&B logging. Use Colab's stop button to interrupt.
RESUME_CHECKPOINT = (
    (repo_model.value if source_location.value == "repo" else source_model.value)
    if resume_mode.value else ""
)
if resume_mode.value:
    assert RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).is_file(), "Select an existing checkpoint in cell 5."
    if source_location.value == "repo":
        assert TASK in HEIGHTMAP_TASKS, "Repository models require a G1-*-HeightMap task."
print(f"Training {TASK} from {RESUME_CHECKPOINT or 'scratch'}", flush=True)
from google.colab import userdata
train_env = {
    **os.environ, "WANDB_API_KEY": userdata.get("WANDB_TOKEN"), "WANDB_MODE": "online",
    "WANDB_X_DISABLE_STATS": "false", "WANDB_X_STATS_SAMPLING_INTERVAL": "5",
}
if WANDB_ENTITY:
    train_env["WANDB_USERNAME"] = WANDB_ENTITY  # The installed RSL-RL writer reads this name.
command = [
    PYTHON, "-u", str(REPO_DIR / "training/scripts/train.py"), TASK,
    "--gpu-ids", "[0]",
    "--env.scene.num-envs", str(NUM_ENVS),
    "--agent.max-iterations", str(ITERATIONS),
    "--agent.num-steps-per-env", str(ROLLOUT_STEPS),
    "--agent.algorithm.num-mini-batches", str(NUM_MINIBATCHES),
    "--agent.save-interval", str(SAVE_INTERVAL),
    "--agent.logger", "wandb",
    "--agent.wandb-project", WANDB_PROJECT,
    "--agent.run-name", RUN_NAME,
]
if START_TERRAIN_LEVEL >= 0:
    command += ["--start-terrain-level", str(START_TERRAIN_LEVEL)]
if RESUME_CHECKPOINT:
    assert Path(RESUME_CHECKPOINT).is_file(), "Resume checkpoint not found."
    command += ["--from", RESUME_CHECKPOINT]
process = subprocess.Popen(
    command, cwd=WORK_DIR, env=train_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    process.send_signal(2)  # Forward Colab's interrupt to the trainer.
    process.wait()
    raise
if return_code:
    raise subprocess.CalledProcessError(return_code, command)


In [ ]:
# 7. Find recent checkpoints. Rerun cell 5 to select one for continuation.
checkpoints = sorted(WORK_DIR.glob("logs/rsl_rl/*/*/model_*.pt"), key=lambda p: p.stat().st_mtime)
for checkpoint in checkpoints[-10:]:
    print(checkpoint)
print("Each run folder also contains TensorBoard logs, policy.onnx, and params/ configs. Cloud metrics are in your W&B project.")
